# memId list linked to 4137846 tax id

## create foruniprot.tar.gz and submit the files to uniprot

In [39]:
%%bash

#Download AFESM clusters mapping file
curl -C - -O https://afesm.steineggerlab.workers.dev/v1/3-repId_memId_cluFlag_taxId_biomeId.tsv.gz
gunzip 3-repId_memId_cluFlag_taxId_biomeId.tsv.gz

# memId list linked to gtdb tax id 4137846:
egrep 4137846 3-repId_memId_cluFlag_taxId_biomeId.tsv -w |  cut -f 2 > memId_with_gtdbtaxid4137846.txt #| egrep MGYP -v

# total proteins
wc memId_with_gtdbtaxid4137846.txt

# number of MGnify proteins
egrep MGYP memId_with_gtdbtaxid4137846.txt | wc

# memId list linked to gtdb tax id 4137846 (only Uniprot):
egrep MGYP memId_with_gtdbtaxid4137846.txt -v > memId_with_gtdbtaxid4137846.txt2
mv memId_with_gtdbtaxid4137846.txt2 memId_with_gtdbtaxid4137846.txt

# number of Uniprot proteins
egrep MGYP memId_with_gtdbtaxid4137846.txt -v | wc

#split to submit to uniprot
mkdir foruniprot
split -l 100000 memId_with_gtdbtaxid4137846.txt memId_with_gtdbtaxid4137846.txt_split_
mv memId_with_gtdbtaxid4137846.txt_split_a* foruniprot/
tar -czvf foruniprot.tar.gz foruniprot/

 1035956  1035956 13110246 memId_with_gtdbtaxid4137846.txt
 297239  297239 5053063
 738717  738717 8057183
foruniprot/
foruniprot/memId_with_gtdbtaxid4137846.txt_split_ae
foruniprot/memId_with_gtdbtaxid4137846.txt_split_ac
foruniprot/memId_with_gtdbtaxid4137846.txt_split_ag
foruniprot/memId_with_gtdbtaxid4137846.txt_split_ah
foruniprot/memId_with_gtdbtaxid4137846.txt_split_aa
foruniprot/memId_with_gtdbtaxid4137846.txt_split_ad
foruniprot/memId_with_gtdbtaxid4137846.txt_split_ab
foruniprot/memId_with_gtdbtaxid4137846.txt_split_af


## this is the memId and lineage mapping file generated by uniprot

In [42]:
!wget -q https://raw.githubusercontent.com/JJPierellaKarlusich/afesm-asgard-eukaryotic-clusters/main/data/taxon4137846/memId_with_gtdbtaxid4137846.Uniprot.tsv.gz
!gunzip -f memId_with_gtdbtaxid4137846.Uniprot.tsv.gz

In [43]:
!egrep -e Promethearchaeati -e Asgard -e Panguiarchaeales memId_with_gtdbtaxid4137846.Uniprot.tsv -i  | cut -f2,4-6 | head

A0A843BYL5	cellular organisms (no rank), Archaea (domain), Promethearchaeati (kingdom), Candidatus Heimdallarchaeota (phylum), unclassified Candidatus Heimdallarchaeota (no rank)	Candidatus Heimdallarchaeota archaeon	2026747
A0A523XQ45	cellular organisms (no rank), Archaea (domain), Promethearchaeati (kingdom), Candidatus Heimdallarchaeota (phylum), unclassified Candidatus Heimdallarchaeota (no rank)	Candidatus Heimdallarchaeota archaeon	2026747
A0A497QQW2	cellular organisms (no rank), Archaea (domain), Promethearchaeati (kingdom), Candidatus Gerdarchaeota (phylum), unclassified Candidatus Gerdarchaeota (no rank)	Candidatus Gerdarchaeota archaeon	2795491
A0A497R265	cellular organisms (no rank), Archaea (domain), Promethearchaeati (kingdom), Candidatus Gerdarchaeota (phylum), unclassified Candidatus Gerdarchaeota (no rank)	Candidatus Gerdarchaeota archaeon	2795491
A0A497R2E6	cellular organisms (no rank), Archaea (domain), Promethearchaeati (kingdom), Candidatus Heimdallarchaeota (phylum

In [48]:
!egrep -e Promethearchaeati -e Asgard -e Panguiarchaeales memId_with_gtdbtaxid4137846.Uniprot.tsv -i  > AsgardAFmemId_with_gtdbtaxid4137846.txt
!wc AsgardAFmemId_with_gtdbtaxid4137846.txt -l
!cut -f4-6 AsgardAFmemId_with_gtdbtaxid4137846.txt | sort | uniq -c | sort -nr

18659 AsgardAFmemId_with_gtdbtaxid4137846.txt
  15238 cellular organisms (no rank), Archaea (domain), Promethearchaeati (kingdom), Candidatus Heimdallarchaeota (phylum), unclassified Candidatus Heimdallarchaeota (no rank)	Candidatus Heimdallarchaeota archaeon	2026747
   3421 cellular organisms (no rank), Archaea (domain), Promethearchaeati (kingdom), Candidatus Gerdarchaeota (phylum), unclassified Candidatus Gerdarchaeota (no rank)	Candidatus Gerdarchaeota archaeon	2795491


# Deleted Ids
### no asgard sequences among them

In [49]:
# List of deleted Uniprot IDs
!egrep deleted memId_with_gtdbtaxid4137846.Uniprot.tsv -i | cut -f1 > deleted_memId.txt

In [50]:
# Retrieve the NCBI tax id and lineage for the deleted Uniprot IDs

import requests
import re
import time

# Read UniProt IDs from a file or stdin output (e.g., piped from shell)
with open("deleted_memId.txt", "r") as f:
    uniprot_ids = [line.strip() for line in f if line.strip()]

# Output file to store results
with open("deleted_memId_with_taxids.tsv", "w") as out_f:
    out_f.write("UniProtID\tNCBI_TaxID\tLineage\n")
    for uid in uniprot_ids:
        url = f"https://rest.uniprot.org/unisave/{uid}?format=txt&versions=1"
        try:
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                match = re.search(r'NCBI_TaxID=(\d+)', response.text)
                taxid = match.group(1) if match else "NA"
                oc = " ".join(re.findall(r'^OC\s+(.*)$', response.text, re.M)).rstrip(".")
                lineage = oc.replace("; ", ";") if oc else "NA"
            else:
                taxid, lineage = "ERROR", "NA"
        except Exception as e:
            taxid, lineage = "ERROR", "NA"
        out_f.write(f"{uid}\t{taxid}\t{lineage}\n")
        time.sleep(0.5)  # polite delay to avoid being rate-limited

### no asgard among the deleted sequences

In [ ]:
!cut -f2,3 deleted_memId_with_taxids.tsv | sort | uniq -c

      1 2026714	Archaea;Candidatus Bathyarchaeota
  13169 2026714	Archaea;Candidatus Bathyarchaeota
   3788 2026795	Archaea;Thaumarchaeota;unclassified Thaumarchaeota
      1 2026795	Archaea;Thermoproteati;Nitrososphaerota;unclassified Nitrososphaerota
   6237 2056631	Archaea;Crenarchaeota
   5379 2093792	Archaea;Candidatus Aenigmarchaeota
   8095 2250254	Archaea;Euryarchaeota;Thermococci
   7632 2250255	Archaea;Euryarchaeota;Stenosarchaea group;Methanomicrobia;Methanosarcinales
   2196 2250274	Archaea;Candidatus Micrarchaeota
   1471 2268195	Archaea;Euryarchaeota;Stenosarchaea group;Methanomicrobia;unclassified Methanomicrobia
   2244 2594798	Archaea;Euryarchaeota;Stenosarchaea group;Candidatus Nanohaloarchaeota;Nanohaloarchaea
      1 NCBI_TaxID	Lineage
